In [ ]:
import torch
import matplotlib.pyplot as plt

In [ ]:
from pfs.ga.bayesian import Model
from pfs.ga.bayesian.distributions import Normal, Uniform
from pfs.ga.bayesian.proposals import NormalProposal, MultivariateNormalProposal
from pfs.ga.bayesian import MCMC
from pfs.ga.bayesian.kernels import GibbsKernel

In [ ]:
class JointProposal(Model):

    def __init__(self, N=(1000,)):
        super().__init__()

        self.N = N

    def model(self, context):
        N = self.N

        # Population-level parameters (outside plate)
        mu = context.sample('mu', Normal(0.0, 5.0, validate_args=False))
        sigma = context.sample('sigma', Uniform(0.1, 3.0, validate_args=False))

        # Member-level latent variable and observation (inside plate)
        with context.plate('n', N):
            x = context.sample('x', Normal(mu, sigma, validate_args=False))
            obs = context.sample('obs', Normal(x, 0.25, validate_args=False), observed=True)

    def step(self, context):
        # Joint step for population parameters [mu, sigma]
        context.step(
            'theta',
            [ self.mu, self.sigma ],
            proposal = MultivariateNormalProposal(
                torch.zeros(self.mu.shape(context.state) + (2,)),
                (torch.eye(2) * 0.5).expand(self.mu.shape(context.state) + (2, 2))
            )
        )

        # Member-level latent variable step
        context.step(
            'x',
            [ self.x ],
            proposal = NormalProposal(
                torch.zeros(self.x.shape(context.state)),
                torch.ones(self.x.shape(context.state)) * 0.5
            )
        )

In [ ]:
model = JointProposal()
model.build()

init_state = model.sample()
observed = { 'obs': init_state['obs'].clone() }

In [ ]:
# Print the hyperparameters
print('mu:', model.mu.value(init_state))
print('sigma:', model.sigma.value(init_state))

In [ ]:
# Plot the distribution of observed data
hist, bins = torch.histogram(observed['obs'], bins=30, density=True)
plt.step(bins[:-1], hist, where='post')

plt.axvline(model.mu.value(init_state), color='red', linestyle='--')
plt.axvline(model.mu.value(init_state) - model.sigma.value(init_state), color='red', linestyle='--')
plt.axvline(model.mu.value(init_state) + model.sigma.value(init_state), color='red', linestyle='--')

In [ ]:
kernel = GibbsKernel(model)
mcmc = MCMC(kernel,
            num_warmup=10000, num_samples=10000, num_chains=10,
            thinning=100,
            progress=True)

In [ ]:
mcmc.run(observed=observed)

In [ ]:
mcmc.trace['mu'].shape, mcmc.trace['sigma'].shape

In [ ]:
print('mu:', init_state['mu'])
print('mu:', torch.mean(mcmc.trace['mu'], dim=0))
print('sigma:', init_state['sigma'])
print('sigma:', torch.mean(mcmc.trace['sigma'], dim=0))

In [ ]:
for i in range(mcmc.trace['mu'].shape[-1]):
    hist, bins = torch.histogram(mcmc.trace['mu'][:, i].flatten(), bins=30, density=True)
    plt.step(bins[:-1], hist, where='post')

plt.axvline(init_state['mu'], color='red', linestyle='--', label='True mu')

In [ ]:
for i in range(mcmc.trace['sigma'].shape[-1]):
    hist, bins = torch.histogram(mcmc.trace['sigma'][:, i].flatten(), bins=30, density=True)
    plt.step(bins[:-1], hist, where='post')

plt.axvline(init_state['sigma'], color='red', linestyle='--', label='True sigma')

In [ ]:
for i in range(mcmc.trace['mu'].shape[-1]):
    plt.plot(mcmc.trace['mu'][..., i], '.')

plt.axhline(init_state['mu'], color='red', linestyle='--', label='True mu')

In [ ]:
for i in range(mcmc.trace['sigma'].shape[-1]):
    plt.plot(mcmc.trace['sigma'][..., i], '.')

plt.axhline(init_state['sigma'], color='red', linestyle='--', label='True sigma')

In [ ]:
mcmc.trace['x'].shape, observed['obs'].shape

In [ ]:
k = 0
for i in range(mcmc.trace['x'].shape[-1]):
    plt.plot(mcmc.trace['x'][:, k, i], '.')

plt.axhline(observed['obs'][k], color='red', linestyle='--', label='True theta')

In [ ]:
init_state['x'].shape, mcmc.trace['x'].shape

In [ ]:
init_state['x'][:10]

In [ ]:
torch.mean(mcmc.trace['x'], dim=(0, -1))[:10]